In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"akhilinyvijeyan","key":"579dd1f2a9f82810fbeccba21da25d4f"}'}

In [4]:
import os
import shutil


os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)


!kaggle datasets download -d abdallahalidev/plantvillage-dataset
print("✅ Download complete!")

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
 99% 2.03G/2.04G [00:16<00:00, 250MB/s]
100% 2.04G/2.04G [00:18<00:00, 117MB/s]
✅ Download complete!


In [5]:

import zipfile
import os


print("Unzipping...")
with zipfile.ZipFile("plantvillage-dataset.zip", 'r') as z:
    z.extractall("plantvillage")
print("✅ Unzipped!")


data_dir = None
for root, dirs, files in os.walk("plantvillage"):
    if len(dirs) > 30:
        data_dir = root
        print(f"\n✅ Found dataset at: {root}")
        print(f"📁 Number of classes: {len(dirs)}")
        print(f"\nFirst 10 classes:")
        for d in sorted(dirs)[:10]:
            count = len(os.listdir(os.path.join(root, d)))
            print(f"  {d}: {count} images")
        break

Unzipping...
✅ Unzipped!

✅ Found dataset at: plantvillage/plantvillage dataset/color
📁 Number of classes: 38

First 10 classes:
  Apple___Apple_scab: 630 images
  Apple___Black_rot: 621 images
  Apple___Cedar_apple_rust: 275 images
  Apple___healthy: 1645 images
  Blueberry___healthy: 1502 images
  Cherry_(including_sour)___Powdery_mildew: 1052 images
  Cherry_(including_sour)___healthy: 854 images
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513 images
  Corn_(maize)___Common_rust_: 1192 images
  Corn_(maize)___Northern_Leaf_Blight: 985 images


In [6]:
# Step 4 — Install and import everything
!pip install -q timm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import timm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import time
import copy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

data_dir = "plantvillage/plantvillage dataset/color"


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


full_dataset = datasets.ImageFolder(data_dir)
class_names  = full_dataset.classes
num_classes  = len(class_names)
print(f"✅ Classes: {num_classes}")
print(f"✅ Total images: {len(full_dataset)}")


with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("✅ Saved class_names.json")


train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)


train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform   = val_transforms

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32,
                          shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print("✅ Data ready!")

✅ Using device: cuda
✅ Classes: 38
✅ Total images: 54305
✅ Saved class_names.json
✅ Train: 43444 | Val: 10861
✅ Data ready!


In [ ]:

model = timm.create_model('efficientnet_b0', pretrained=True,
                           num_classes=num_classes)
model = model.to(device)
print("✅ EfficientNetB0 loaded with pretrained ImageNet weights")


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


def train_model(model, train_loader, val_loader, epochs=10):
    best_acc   = 0.0
    best_model = None
    history    = {'train_loss': [], 'val_loss': [],
                  'train_acc':  [], 'val_acc':  []}

    for epoch in range(epochs):
        start = time.time()


        model.train()
        train_loss, train_correct = 0.0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss    += loss.item() * images.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()


        model.eval()
        val_loss, val_correct = 0.0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs  = model(images)
                loss     = criterion(outputs, labels)
                val_loss    += loss.item() * images.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()


        train_loss /= len(train_dataset)
        val_loss   /= len(val_dataset)
        train_acc   = train_correct / len(train_dataset)
        val_acc     = val_correct   / len(val_dataset)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        scheduler.step()
        elapsed = time.time() - start


        if val_acc > best_acc:
            best_acc   = val_acc
            best_model = copy.deepcopy(model.state_dict())
            torch.save(best_model, "cropshield_best.pth")
            tag = "⭐ BEST"
        else:
            tag = ""

        print(f"Epoch [{epoch+1:02d}/{epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"Time: {elapsed:.1f}s {tag}")

    print(f"\n🏆 Best Validation Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
    return history


print("\n🚀 Starting training — 10 epochs on Tesla T4 GPU...")
print("Expected time: ~25-30 minutes\n")
history = train_model(model, train_loader, val_loader, epochs=10)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

✅ EfficientNetB0 loaded with pretrained ImageNet weights

🚀 Starting training — 10 epochs on Tesla T4 GPU...
Expected time: ~25-30 minutes

Epoch [01/10] Train Loss: 0.1736 Acc: 0.9488 | Val Loss: 0.0958 Acc: 0.9693 | Time: 263.3s ⭐ BEST
Epoch [02/10] Train Loss: 0.0691 Acc: 0.9777 | Val Loss: 0.0984 Acc: 0.9715 | Time: 262.5s ⭐ BEST
Epoch [03/10] Train Loss: 0.0577 Acc: 0.9817 | Val Loss: 0.1284 Acc: 0.9760 | Time: 263.3s ⭐ BEST
Epoch [04/10] Train Loss: 0.0443 Acc: 0.9860 | Val Loss: 0.0340 Acc: 0.9899 | Time: 266.0s ⭐ BEST
